In [2]:
import os
import hashlib
import pandas as pd

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

INPUT_FILE = "train_balanced.csv"

OUTPUT_DIR = "digikala_rating_sampling_v1"

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "digikala_rating_semi_supervised_600k_v1.csv"
)

TEXT_COL = "text"
RATE_COL = "rate"

SAMPLES_PER_CLASS = 200_000
RANDOM_STATE = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --------------------------------------------------
# 2. Load data
# --------------------------------------------------

df = pd.read_csv(INPUT_FILE)

required_columns = [TEXT_COL, RATE_COL]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

print("Original dataset:", len(df))

# Stable ID based on original row position
if "row_id" in df.columns:
    raise ValueError(
        "The input already contains row_id. "
        "Choose a different ID column or remove this check."
    )

df = df.reset_index(drop=True)

df["row_id"] = [
    f"source_row_{i:09d}"
    for i in range(len(df))
]

# --------------------------------------------------
# 3. Clean rate values
# --------------------------------------------------

raw_rate = df[RATE_COL].copy()

df[RATE_COL] = pd.to_numeric(
    raw_rate,
    errors="coerce"
)

# Distinguish truly missing values from malformed values
originally_missing = raw_rate.isna() | (
    raw_rate.astype("string").str.strip().eq("").fillna(False)
)

numeric_zero = df[RATE_COL].eq(0)

valid_rates = df[RATE_COL].isin([1, 2, 3, 4, 5])

malformed_rate = (
    ~originally_missing
    & ~numeric_zero
    & ~valid_rates
)

# --------------------------------------------------
# 4. Assign rating bands
# --------------------------------------------------

df["rating_band"] = pd.NA

df.loc[
    df[RATE_COL].isin([1, 2]),
    "rating_band"
] = "low"

df.loc[
    df[RATE_COL] == 3,
    "rating_band"
] = "medium"

df.loc[
    df[RATE_COL].isin([4, 5]),
    "rating_band"
] = "high"

# --------------------------------------------------
# 5. Assign label status
# --------------------------------------------------

df["label_status"] = "invalid"

df.loc[
    originally_missing | numeric_zero,
    "label_status"
] = "unlabeled"

df.loc[
    valid_rates,
    "label_status"
] = "labeled"

# --------------------------------------------------
# 6. Remove empty text and invalid rates
# --------------------------------------------------

df[TEXT_COL] = (
    df[TEXT_COL]
    .fillna("")
    .astype(str)
    .str.strip()
)

empty_text = df[TEXT_COL].eq("")

quality_report = {
    "original_rows": int(len(df)),
    "empty_text_rows": int(empty_text.sum()),
    "missing_rate_rows": int(originally_missing.sum()),
    "zero_rate_rows": int(numeric_zero.sum()),
    "malformed_rate_rows": int(malformed_rate.sum()),
    "valid_labeled_rows": int(valid_rates.sum())
}

df = df[
    ~empty_text
    & ~malformed_rate
].copy()

# --------------------------------------------------
# 7. Separate labeled classes
# --------------------------------------------------

class_data = {
    "low": df[df["rating_band"] == "low"].copy(),
    "medium": df[df["rating_band"] == "medium"].copy(),
    "high": df[df["rating_band"] == "high"].copy()
}

# --------------------------------------------------
# 8. Check class sizes
# --------------------------------------------------

for class_name, class_df in class_data.items():

    print(f"{class_name}: {len(class_df):,}")

    if len(class_df) < SAMPLES_PER_CLASS:
        raise ValueError(
            f"Not enough samples in {class_name}: "
            f"{len(class_df):,} available, "
            f"{SAMPLES_PER_CLASS:,} required."
        )

# --------------------------------------------------
# 9. Random sampling
# --------------------------------------------------

sampled_classes = []

for class_name, class_df in class_data.items():

    sampled = class_df.sample(
        n=SAMPLES_PER_CLASS,
        replace=False,
        random_state=RANDOM_STATE
    )

    sampled_classes.append(sampled)

# --------------------------------------------------
# 10. Combine labeled samples
# --------------------------------------------------

labeled_data = pd.concat(
    sampled_classes,
    ignore_index=True
)

# --------------------------------------------------
# 11. Add all naturally unlabeled comments
# --------------------------------------------------

unlabeled_data = df[
    df["label_status"] == "unlabeled"
].copy()

unlabeled_data["unlabeled_source"] = "missing_or_zero_rate"

# --------------------------------------------------
# 12. Combine labeled and unlabeled data
# --------------------------------------------------

final_data = pd.concat(
    [labeled_data, unlabeled_data],
    ignore_index=True
)

# --------------------------------------------------
# 13. Encode labels
# --------------------------------------------------

label_mapping = {
    "low": 0,
    "medium": 1,
    "high": 2
}

final_data["labels"] = (
    final_data["rating_band"]
    .map(label_mapping)
    .astype("Int64")
)

# --------------------------------------------------
# 14. Shuffle final dataset
# --------------------------------------------------

final_data = final_data.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

# --------------------------------------------------
# 15. Check duplicate IDs
# --------------------------------------------------

if final_data["row_id"].duplicated().any():
    raise ValueError("Duplicate row_id detected!")

# --------------------------------------------------
# 16. Save output
# --------------------------------------------------

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        f"Output already exists: {OUTPUT_FILE}"
    )

final_data.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# --------------------------------------------------
# 17. Final report
# --------------------------------------------------

print("\n" + "=" * 55)
print("RATING SAMPLING COMPLETED")
print("=" * 55)

print("\nQuality report:")
for key, value in quality_report.items():
    print(f"{key}: {value:,}")

print("\nLabeled class distribution:")

print(
    final_data["rating_band"]
    .value_counts()
    .reindex(
        ["low", "medium", "high"],
        fill_value=0
    )
)

print("\nUnlabeled samples:", len(unlabeled_data))

print("Final dataset size:", len(final_data))

print("\nLabel mapping:", label_mapping)

print("\nSaved to:", OUTPUT_FILE)


Original dataset: 3639394
low: 366,427
medium: 587,746
high: 2,116,049

RATING SAMPLING COMPLETED

Quality report:
original_rows: 3,639,394
empty_text_rows: 0
missing_rate_rows: 0
zero_rate_rows: 395,816
malformed_rate_rows: 173,356
valid_labeled_rows: 3,070,222

Labeled class distribution:
rating_band
low       200000
medium    200000
high      200000
Name: count, dtype: int64

Unlabeled samples: 395816
Final dataset size: 995816

Label mapping: {'low': 0, 'medium': 1, 'high': 2}

Saved to: digikala_rating_sampling_v1\digikala_rating_semi_supervised_600k_v1.csv


In [3]:

# ============================================================
# Digikala Rating Classification
# Semi-Supervised Learning with ParsBERT
#
# Stage 1: Supervised Training
# Stage 2: Pseudo Labeling
# Stage 3: Semi-Supervised Fine-Tuning
# Stage 4: Independent Test Evaluation
#
# Transformers: 5.21.0
# ============================================================


# ============================================================
# 1. Imports
# ============================================================

import os
import gc
import json
import math
import random
import hashlib
import inspect
import warnings

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    f1_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed
)

warnings.filterwarnings("ignore")


# ============================================================
# 2. Configuration
# ============================================================

# ------------------------------------------------------------
# Input files
# ------------------------------------------------------------

INPUT_FILE = (
    "digikala_rating_sampling_v1/"
    "digikala_rating_semi_supervised_600k_v1.csv"
)

TEST_FILE = "test_not_mismatch_rated.csv"


# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------

OUTPUT_ROOT = "digikala_parsbert_semi_supervised"

SPLIT_DIR = os.path.join(OUTPUT_ROOT, "splits")

STAGE1_DIR = os.path.join(OUTPUT_ROOT, "stage1_supervised")

STAGE2_DIR = os.path.join(OUTPUT_ROOT, "stage2_semi_supervised")

PSEUDO_DIR = os.path.join(OUTPUT_ROOT, "pseudo_labels")

EVALUATION_DIR = os.path.join(OUTPUT_ROOT, "evaluation")

for directory in [
    OUTPUT_ROOT,
    SPLIT_DIR,
    STAGE1_DIR,
    STAGE2_DIR,
    PSEUDO_DIR,
    EVALUATION_DIR
]:
    os.makedirs(directory, exist_ok=True)


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

MODEL_NAME = "HooshvareLab/bert-base-parsbert-uncased"

MAX_LENGTH = 128

NUM_LABELS = 3

LABEL2ID = {
    "low": 0,
    "medium": 1,
    "high": 2
}

ID2LABEL = {
    0: "low",
    1: "medium",
    2: "high"
}


# ------------------------------------------------------------
# Dataset columns
# ------------------------------------------------------------

TEXT_COL = "text"

ROW_ID_COL = "row_id"

TEST_ID_COL = "id"

LABEL_COL = "labels"

RATING_BAND_COL = "rating_band"

RATE_COL = "rate"


# ------------------------------------------------------------
# Split configuration
# ------------------------------------------------------------

SPLIT_SEED = 2026

TRAIN_RATIO = 0.90

MASK_RATE = 0.20

MASK_SEED = "digikala-semi-supervised-v1"


# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

EPOCHS = 3

TRAIN_BATCH_SIZE = 32

EVAL_BATCH_SIZE = 64

INFERENCE_BATCH_SIZE = 128

LEARNING_RATE = 2e-5

WEIGHT_DECAY = 0.01

WARMUP_RATIO = 0.06

LABEL_SMOOTHING = 0.1

EVAL_STEPS = 10000

SAVE_STEPS = 10000

LOGGING_STEPS = 500

EARLY_STOPPING_PATIENCE = 3


# ------------------------------------------------------------
# Semi-supervised configuration
# ------------------------------------------------------------

CONFIDENCE_THRESHOLD = 0.95

# Optional: maximum pseudo-label count per class.
# None means no limit.

MAX_PSEUDO_PER_CLASS = None


# ------------------------------------------------------------
# Hardware
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_FP16 = torch.cuda.is_available()

print("=" * 65)
print("DIGIKALA PARSBERT SEMI-SUPERVISED TRAINING")
print("=" * 65)

print("Device:", DEVICE)
print("Transformers version:", __import__("transformers").__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 3. Reproducibility
# ============================================================

def seed_everything(seed=SPLIT_SEED):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    set_seed(seed)


seed_everything()


# ============================================================
# 4. Stable Hash
# ============================================================

def stable_hash(row_id):

    key = f"{MASK_SEED}|{row_id}".encode("utf-8")

    return int(
        hashlib.sha256(key).hexdigest()[:16],
        16
    )


# ============================================================
# 5. Label Conversion
# ============================================================

def normalize_class_label(value):

    """
    Converts existing class labels into:
        low    -> 0
        medium -> 1
        high   -> 2

    Also supports existing numeric class IDs 0, 1, 2.
    """

    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    if value in LABEL2ID:
        return LABEL2ID[value]

    try:

        numeric_value = float(value)

        if numeric_value in [0, 1, 2]:
            return int(numeric_value)

    except (ValueError, TypeError):
        pass

    return np.nan


def normalize_original_rating(value):

    """
    Converts original numerical ratings:

        1, 2 -> low
        3    -> medium
        4, 5 -> high

    Also accepts existing low/medium/high labels.
    """

    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    if value in LABEL2ID:
        return LABEL2ID[value]

    try:

        rating = float(value)

        if rating in [1, 2]:
            return 0

        if rating == 3:
            return 1

        if rating in [4, 5]:
            return 2

    except (ValueError, TypeError):
        pass

    return np.nan


# ============================================================
# 6. Load Training Dataset
# ============================================================

print("\n" + "=" * 65)
print("LOADING TRAINING DATA")
print("=" * 65)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(INPUT_FILE)

data = pd.read_csv(INPUT_FILE)

print("Original shape:", data.shape)

required_columns = [TEXT_COL]

for column in required_columns:

    if column not in data.columns:
        raise ValueError(
            f"Required column not found: {column}"
        )


# ------------------------------------------------------------
# Row ID
# ------------------------------------------------------------

if ROW_ID_COL not in data.columns:

    if "id" in data.columns:

        data[ROW_ID_COL] = data["id"]

    else:

        data[ROW_ID_COL] = np.arange(len(data))


if data[ROW_ID_COL].isna().any():

    raise ValueError(
        "row_id contains missing values."
    )


data[ROW_ID_COL] = data[ROW_ID_COL].astype(str)


if data[ROW_ID_COL].duplicated().any():

    duplicate_count = data[ROW_ID_COL].duplicated().sum()

    raise ValueError(
        f"Duplicate row_id values found: {duplicate_count}"
    )


# ------------------------------------------------------------
# Text
# ------------------------------------------------------------

data[TEXT_COL] = (
    data[TEXT_COL]
    .fillna("")
    .astype(str)
    .str.strip()
)

data = data[
    data[TEXT_COL].ne("")
].copy()


# ------------------------------------------------------------
# Resolve existing training labels
# ------------------------------------------------------------

if LABEL_COL in data.columns:

    data[LABEL_COL] = data[LABEL_COL].apply(
        normalize_class_label
    )

elif RATING_BAND_COL in data.columns:

    data[LABEL_COL] = data[RATING_BAND_COL].apply(
        normalize_class_label
    )

elif RATE_COL in data.columns:

    data[LABEL_COL] = data[RATE_COL].apply(
        normalize_original_rating
    )

else:

    raise ValueError(
        "No usable label column found. "
        "Expected labels, rating_band, or rate."
    )


data[LABEL_COL] = pd.to_numeric(
    data[LABEL_COL],
    errors="coerce"
)


# ------------------------------------------------------------
# Separate labeled and naturally unlabeled data
# ------------------------------------------------------------

labeled_data = data[
    data[LABEL_COL].isin([0, 1, 2])
].copy()

natural_unlabeled = data[
    ~data[LABEL_COL].isin([0, 1, 2])
].copy()

labeled_data[LABEL_COL] = (
    labeled_data[LABEL_COL].astype(int)
)


print("\nLabeled samples:", len(labeled_data))

print("Naturally unlabeled samples:", len(natural_unlabeled))

print("\nLabeled class distribution:")

print(
    labeled_data[LABEL_COL]
    .map(ID2LABEL)
    .value_counts()
    .reindex(["low", "medium", "high"], fill_value=0)
)


if labeled_data.empty:

    raise ValueError(
        "No labeled training samples found."
    )


# ============================================================
# 7. Train / Validation Split
# ============================================================

print("\n" + "=" * 65)
print("TRAIN / VALIDATION SPLIT")
print("=" * 65)

train_pool, validation_df = train_test_split(
    labeled_data,
    train_size=TRAIN_RATIO,
    random_state=SPLIT_SEED,
    stratify=labeled_data[LABEL_COL]
)

train_pool = train_pool.copy()

validation_df = validation_df.copy()

train_pool.reset_index(drop=True, inplace=True)

validation_df.reset_index(drop=True, inplace=True)


print("Train pool:", len(train_pool))

print("Validation:", len(validation_df))


print("\nTrain pool distribution:")

print(
    train_pool[LABEL_COL]
    .map(ID2LABEL)
    .value_counts()
    .reindex(["low", "medium", "high"], fill_value=0)
)


print("\nValidation distribution:")

print(
    validation_df[LABEL_COL]
    .map(ID2LABEL)
    .value_counts()
    .reindex(["low", "medium", "high"], fill_value=0)
)


# ============================================================
# 8. Deterministic 20% Label Masking
# ============================================================

print("\n" + "=" * 65)
print("DETERMINISTIC LABEL MASKING")
print("=" * 65)

train_pool["mask_hash"] = train_pool[
    ROW_ID_COL
].apply(stable_hash)


train_pool["is_masked"] = False

mask_manifest = []

mask_summary = {}


for class_id, class_name in ID2LABEL.items():

    class_indices = train_pool.index[
        train_pool[LABEL_COL] == class_id
    ].tolist()

    class_df = train_pool.loc[
        class_indices
    ].copy()

    class_df = class_df.sort_values(
        "mask_hash"
    )

    class_count = len(class_df)

    mask_count = math.floor(
        MASK_RATE * class_count
    )

    selected_indices = class_df.index[
        :mask_count
    ].tolist()

    train_pool.loc[
        selected_indices,
        "is_masked"
    ] = True

    for index in selected_indices:

        mask_manifest.append({

            ROW_ID_COL: train_pool.loc[
                index, ROW_ID_COL
            ],

            "mask_hash": train_pool.loc[
                index, "mask_hash"
            ],

            "mask_status": "masked",

            "class_name_for_audit": class_name

        })

    mask_summary[class_name] = {

        "original_count": int(class_count),

        "masked_count": int(mask_count),

        "remaining_labeled": int(
            class_count - mask_count
        ),

        "actual_mask_rate": (
            mask_count / class_count
            if class_count > 0 else 0
        )

    }


mask_manifest_df = pd.DataFrame(mask_manifest)

mask_manifest_df.to_csv(
    os.path.join(
        SPLIT_DIR,
        "mask_manifest.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# Create supervised and masked datasets
# ------------------------------------------------------------

supervised_train = train_pool[
    ~train_pool["is_masked"]
].copy()

masked_train = train_pool[
    train_pool["is_masked"]
].copy()


# ------------------------------------------------------------
# Hide masked labels
# ------------------------------------------------------------

masked_train[LABEL_COL] = np.nan

masked_train[RATING_BAND_COL] = (
    masked_train[RATING_BAND_COL]
    if RATING_BAND_COL in masked_train.columns
    else np.nan
)


# ------------------------------------------------------------
# Build pseudo-label candidates
# ------------------------------------------------------------

pseudo_candidates = pd.concat(
    [
        masked_train,
        natural_unlabeled
    ],
    ignore_index=True
)

pseudo_candidates = pseudo_candidates.drop_duplicates(
    subset=[ROW_ID_COL]
).reset_index(drop=True)


print("\nMask summary:")

print(
    json.dumps(
        mask_summary,
        ensure_ascii=False,
        indent=4
    )
)


print("\nSupervised training samples:", len(supervised_train))

print("Masked samples:", len(masked_train))

print("Natural unlabeled samples:", len(natural_unlabeled))

print("Total pseudo-label candidates:", len(pseudo_candidates))


# ------------------------------------------------------------
# Save splits
# ------------------------------------------------------------

supervised_train.to_csv(
    os.path.join(
        SPLIT_DIR,
        "supervised_train.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

validation_df.to_csv(
    os.path.join(
        SPLIT_DIR,
        "validation.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

pseudo_candidates.to_csv(
    os.path.join(
        SPLIT_DIR,
        "pseudo_candidates.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

with open(
    os.path.join(
        SPLIT_DIR,
        "mask_summary.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        mask_summary,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# 9. Tokenizer
# ============================================================

print("\n" + "=" * 65)
print("LOADING TOKENIZER")
print("=" * 65)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


# ============================================================
# 10. PyTorch Dataset
# ============================================================

class RatingDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length=MAX_LENGTH,
        include_labels=True
    ):

        self.texts = dataframe[
            TEXT_COL
        ].astype(str).tolist()

        self.include_labels = include_labels

        if include_labels:

            self.labels = dataframe[
                LABEL_COL
            ].astype(int).tolist()

        self.tokenizer = tokenizer

        self.max_length = max_length


    def __len__(self):

        return len(self.texts)


    def __getitem__(self, index):

        text = self.texts[index]

        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length
        )

        if self.include_labels:

            encoded["labels"] = self.labels[index]

        return encoded


data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


# ============================================================
# 11. Metrics
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {

        "accuracy": accuracy,

        "macro_f1": macro_f1,

        "weighted_f1": weighted_f1

    }


# ============================================================
# 12. Model Initialization
# ============================================================

def load_model():

    model = AutoModelForSequenceClassification.from_pretrained(

        MODEL_NAME,

        num_labels=NUM_LABELS,

        id2label=ID2LABEL,

        label2id=LABEL2ID

    )

    return model


# ============================================================
# 13. Version-Compatible Trainer
# ============================================================

def create_training_arguments(

    output_dir,

    train_size,

    epochs=EPOCHS

):

    steps_per_epoch = math.ceil(
        train_size / TRAIN_BATCH_SIZE
    )

    total_steps = math.ceil(
        steps_per_epoch * epochs
    )

    warmup_steps = int(
        total_steps * WARMUP_RATIO
    )

    # Dynamic evaluation interval:
    # ensure evaluation happens even for smaller Stage 2 datasets.

    eval_steps = max(
        1,
        min(
            EVAL_STEPS,
            steps_per_epoch
        )
    )

    save_steps = eval_steps

    kwargs = {

        "output_dir": output_dir,

        "num_train_epochs": epochs,

        "per_device_train_batch_size": TRAIN_BATCH_SIZE,

        "per_device_eval_batch_size": EVAL_BATCH_SIZE,

        "learning_rate": LEARNING_RATE,

        "weight_decay": WEIGHT_DECAY,

        "warmup_steps": warmup_steps,

        "label_smoothing_factor": LABEL_SMOOTHING,

        "eval_strategy": "steps",

        "save_strategy": "steps",

        "eval_steps": eval_steps,

        "save_steps": save_steps,

        "logging_strategy": "steps",

        "logging_steps": LOGGING_STEPS,

        "load_best_model_at_end": True,

        "metric_for_best_model": "eval_macro_f1",

        "greater_is_better": True,

        "save_total_limit": 2,

        "fp16": USE_FP16,

        "report_to": "none",

        "seed": SPLIT_SEED,

        "data_seed": SPLIT_SEED,

        "optim": "adamw_torch",

        "dataloader_num_workers": 0,

        "dataloader_pin_memory": torch.cuda.is_available()

    }

    # Keep only parameters supported by installed Transformers.

    supported_parameters = inspect.signature(
        TrainingArguments.__init__
    ).parameters

    kwargs = {

        key: value

        for key, value in kwargs.items()

        if key in supported_parameters

    }

    return TrainingArguments(**kwargs)


def create_trainer(

    model,

    train_dataset,

    eval_dataset,

    output_dir,

    epochs=EPOCHS

):

    training_args = create_training_arguments(

        output_dir=output_dir,

        train_size=len(train_dataset),

        epochs=epochs

    )

    trainer_parameters = inspect.signature(
        Trainer.__init__
    ).parameters

    trainer_kwargs = {

        "model": model,

        "args": training_args,

        "train_dataset": train_dataset,

        "eval_dataset": eval_dataset,

        "data_collator": data_collator,

        "compute_metrics": compute_metrics,

        "callbacks": [

            EarlyStoppingCallback(

                early_stopping_patience=EARLY_STOPPING_PATIENCE

            )

        ]

    }

    # Transformers API compatibility.

    if "processing_class" in trainer_parameters:

        trainer_kwargs["processing_class"] = tokenizer

    elif "tokenizer" in trainer_parameters:

        trainer_kwargs["tokenizer"] = tokenizer

    return Trainer(**trainer_kwargs)


# ============================================================
# 14. Create Stage 1 Datasets
# ============================================================

print("\n" + "=" * 65)
print("PREPARING STAGE 1 DATASETS")
print("=" * 65)

stage1_train_dataset = RatingDataset(
    supervised_train,
    tokenizer,
    include_labels=True
)

validation_dataset = RatingDataset(
    validation_df,
    tokenizer,
    include_labels=True
)


# ============================================================
# 15. Stage 1: Supervised Training
# ============================================================

print("\n" + "=" * 65)
print("STAGE 1: SUPERVISED TRAINING")
print("=" * 65)

model_stage1 = load_model()

trainer_stage1 = create_trainer(

    model=model_stage1,

    train_dataset=stage1_train_dataset,

    eval_dataset=validation_dataset,

    output_dir=STAGE1_DIR,

    epochs=EPOCHS

)

trainer_stage1.train()


# ------------------------------------------------------------
# Save best Stage 1 model
# ------------------------------------------------------------

trainer_stage1.save_model(
    STAGE1_DIR
)

tokenizer.save_pretrained(
    STAGE1_DIR
)


# ------------------------------------------------------------
# Stage 1 validation
# ------------------------------------------------------------

stage1_validation_metrics = trainer_stage1.evaluate()

print("\nStage 1 Validation Metrics:")

print(stage1_validation_metrics)


with open(
    os.path.join(
        STAGE1_DIR,
        "validation_metrics.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        stage1_validation_metrics,
        f,
        indent=4
    )


# ============================================================
# 16. Pseudo-Label Inference
# ============================================================

def predict_probabilities(

    model,

    dataframe,

    tokenizer,

    batch_size=INFERENCE_BATCH_SIZE

):

    if len(dataframe) == 0:

        return (
            np.empty((0, NUM_LABELS)),
            np.empty((0,), dtype=int),
            np.empty((0,))
        )


    dataset = RatingDataset(

        dataframe,

        tokenizer,

        include_labels=False

    )

    loader = DataLoader(

        dataset,

        batch_size=batch_size,

        shuffle=False,

        collate_fn=data_collator,

        num_workers=0,

        pin_memory=torch.cuda.is_available()

    )

    model.eval()

    model.to(DEVICE)

    all_probabilities = []

    with torch.no_grad():

        for batch in loader:

            batch = {

                key: value.to(DEVICE)

                for key, value in batch.items()

            }

            outputs = model(**batch)

            probabilities = torch.softmax(
                outputs.logits,
                dim=-1
            )

            all_probabilities.append(
                probabilities.cpu().numpy()
            )


    all_probabilities = np.concatenate(
        all_probabilities,
        axis=0
    )

    predictions = np.argmax(
        all_probabilities,
        axis=1
    )

    confidences = np.max(
        all_probabilities,
        axis=1
    )

    return (
        all_probabilities,
        predictions,
        confidences
    )


# ============================================================
# 17. Stage 2: Generate Pseudo Labels
# ============================================================

print("\n" + "=" * 65)
print("STAGE 2: PSEUDO LABEL GENERATION")
print("=" * 65)

print(
    "Pseudo-label candidates:",
    len(pseudo_candidates)
)

probabilities, predictions, confidences = predict_probabilities(

    model=trainer_stage1.model,

    dataframe=pseudo_candidates,

    tokenizer=tokenizer

)


pseudo_candidates = pseudo_candidates.copy()

pseudo_candidates["pseudo_label"] = predictions

pseudo_candidates["pseudo_confidence"] = confidences

pseudo_candidates["pseudo_label_name"] = (
    pseudo_candidates["pseudo_label"]
    .map(ID2LABEL)
)


# ------------------------------------------------------------
# Confidence threshold
# ------------------------------------------------------------

accepted_mask = (
    pseudo_candidates["pseudo_confidence"]
    >= CONFIDENCE_THRESHOLD
)

accepted_pseudo = pseudo_candidates[
    accepted_mask
].copy()


# ------------------------------------------------------------
# Optional class-wise cap
# ------------------------------------------------------------

if MAX_PSEUDO_PER_CLASS is not None:

    capped_classes = []

    for class_id in range(NUM_LABELS):

        class_pseudo = accepted_pseudo[
            accepted_pseudo["pseudo_label"] == class_id
        ].copy()

        class_pseudo = class_pseudo.sort_values(
            "pseudo_confidence",
            ascending=False
        )

        class_pseudo = class_pseudo.head(
            MAX_PSEUDO_PER_CLASS
        )

        capped_classes.append(class_pseudo)


    if capped_classes:

        accepted_pseudo = pd.concat(
            capped_classes,
            ignore_index=True
        )


# ------------------------------------------------------------
# Assign accepted pseudo labels
# ------------------------------------------------------------

accepted_pseudo[LABEL_COL] = (
    accepted_pseudo["pseudo_label"].astype(int)
)

accepted_pseudo["is_pseudo"] = True


# ------------------------------------------------------------
# Pseudo-label report
# ------------------------------------------------------------

print("\nPseudo-label results:")

print("Total candidates:", len(pseudo_candidates))

print("Accepted:", len(accepted_pseudo))

print(
    "Acceptance rate:",
    round(
        len(accepted_pseudo) / max(len(pseudo_candidates), 1),
        4
    )
)


print("\nAccepted pseudo-label distribution:")

print(
    accepted_pseudo["pseudo_label"]
    .map(ID2LABEL)
    .value_counts()
    .reindex(["low", "medium", "high"], fill_value=0)
)


# ------------------------------------------------------------
# Save pseudo labels
# ------------------------------------------------------------

accepted_pseudo.to_csv(

    os.path.join(
        PSEUDO_DIR,
        "accepted_pseudo_labels.csv"
    ),

    index=False,

    encoding="utf-8-sig"

)


pseudo_candidates.to_csv(

    os.path.join(
        PSEUDO_DIR,
        "all_pseudo_predictions.csv"
    ),

    index=False,

    encoding="utf-8-sig"

)


pseudo_report = {

    "total_candidates": int(len(pseudo_candidates)),

    "accepted_pseudo_labels": int(len(accepted_pseudo)),

    "confidence_threshold": CONFIDENCE_THRESHOLD,

    "acceptance_rate": float(
        len(accepted_pseudo) /
        max(len(pseudo_candidates), 1)
    ),

    "class_distribution": {

        ID2LABEL[class_id]: int(
            (accepted_pseudo["pseudo_label"] == class_id).sum()
        )

        for class_id in range(NUM_LABELS)

    }

}


with open(

    os.path.join(
        PSEUDO_DIR,
        "pseudo_label_report.json"
    ),

    "w",

    encoding="utf-8"

) as f:

    json.dump(
        pseudo_report,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# 18. Stage 3: Build Semi-Supervised Training Dataset
# ============================================================

print("\n" + "=" * 65)
print("STAGE 3: BUILDING SEMI-SUPERVISED DATASET")
print("=" * 65)


# ------------------------------------------------------------
# True labeled training data
# ------------------------------------------------------------

true_train = supervised_train.copy()

true_train["is_pseudo"] = False


# ------------------------------------------------------------
# Pseudo-labeled data
# ------------------------------------------------------------

pseudo_train = accepted_pseudo.copy()


# ------------------------------------------------------------
# Combine true and pseudo labels
# ------------------------------------------------------------

stage2_train_df = pd.concat(

    [

        true_train,

        pseudo_train

    ],

    ignore_index=True

)


stage2_train_df = stage2_train_df.drop_duplicates(

    subset=[ROW_ID_COL]

).reset_index(drop=True)


stage2_train_df[LABEL_COL] = (
    stage2_train_df[LABEL_COL].astype(int)
)


print("True labeled samples:", len(true_train))

print("Pseudo-labeled samples:", len(pseudo_train))

print("Total Stage 2 samples:", len(stage2_train_df))


print("\nStage 2 class distribution:")

print(

    stage2_train_df[LABEL_COL]

    .map(ID2LABEL)

    .value_counts()

    .reindex(["low", "medium", "high"], fill_value=0)

)


# ------------------------------------------------------------
# Save Stage 2 dataset
# ------------------------------------------------------------

stage2_train_df.to_csv(

    os.path.join(
        STAGE2_DIR,
        "stage2_training_data.csv"
    ),

    index=False,

    encoding="utf-8-sig"

)


# ============================================================
# 19. Stage 3: Semi-Supervised Fine-Tuning
# ============================================================

print("\n" + "=" * 65)
print("STAGE 3: SEMI-SUPERVISED FINE-TUNING")
print("=" * 65)


stage2_train_dataset = RatingDataset(

    stage2_train_df,

    tokenizer,

    include_labels=True

)


# ------------------------------------------------------------
# Initialize Stage 2 from the best Stage 1 model
# ------------------------------------------------------------

model_stage2 = AutoModelForSequenceClassification.from_pretrained(

    STAGE1_DIR

)


trainer_stage2 = create_trainer(

    model=model_stage2,

    train_dataset=stage2_train_dataset,

    eval_dataset=validation_dataset,

    output_dir=STAGE2_DIR,

    epochs=EPOCHS

)


trainer_stage2.train()


# ------------------------------------------------------------
# Save final Stage 2 model
# ------------------------------------------------------------

trainer_stage2.save_model(
    STAGE2_DIR
)

tokenizer.save_pretrained(
    STAGE2_DIR
)


# ------------------------------------------------------------
# Validation evaluation
# ------------------------------------------------------------

stage2_validation_metrics = trainer_stage2.evaluate()

print("\nStage 2 Validation Metrics:")

print(stage2_validation_metrics)


with open(

    os.path.join(
        STAGE2_DIR,
        "validation_metrics.json"
    ),

    "w",

    encoding="utf-8"

) as f:

    json.dump(
        stage2_validation_metrics,
        f,
        indent=4
    )


# ============================================================
# 20. Load Independent Test Dataset
# ============================================================

print("\n" + "=" * 65)
print("LOADING INDEPENDENT TEST DATASET")
print("=" * 65)


if not os.path.exists(TEST_FILE):

    raise FileNotFoundError(TEST_FILE)


test_df = pd.read_csv(TEST_FILE)

print("Original test shape:", test_df.shape)

print("Test columns:", test_df.columns.tolist())


# ------------------------------------------------------------
# Required test columns
# ------------------------------------------------------------

required_test_columns = [

    TEST_ID_COL,

    TEXT_COL,

    RATE_COL

]

missing_test_columns = [

    column

    for column in required_test_columns

    if column not in test_df.columns

]

if missing_test_columns:

    raise ValueError(

        f"Missing test columns: {missing_test_columns}"

    )


# ------------------------------------------------------------
# Keep only required columns
# ------------------------------------------------------------

test_df = test_df[

    [

        TEST_ID_COL,

        TEXT_COL,

        RATE_COL

    ]

].copy()


# ------------------------------------------------------------
# Text
# ------------------------------------------------------------

test_df[TEXT_COL] = (

    test_df[TEXT_COL]

    .fillna("")

    .astype(str)

    .str.strip()

)


# ------------------------------------------------------------
# Existing test labels
#
# Test file:
# rate = low / medium / high
#
# This is only encoding existing labels.
# No labels are generated from rate_score.
# ------------------------------------------------------------

test_df[RATE_COL] = (

    test_df[RATE_COL]

    .astype("string")

    .str.strip()

    .str.lower()

)


test_df[LABEL_COL] = test_df[RATE_COL].map(
    LABEL2ID
)


# ------------------------------------------------------------
# Validate labels
# ------------------------------------------------------------

if test_df[LABEL_COL].isna().any():

    invalid_values = (

        test_df.loc[

            test_df[LABEL_COL].isna(),

            RATE_COL

        ]

        .value_counts(dropna=False)

        .to_dict()

    )

    raise ValueError(

        "Unknown or missing test labels found: "

        f"{invalid_values}"

    )


test_df[LABEL_COL] = (
    test_df[LABEL_COL].astype(int)
)


# ------------------------------------------------------------
# Remove empty texts
# ------------------------------------------------------------

test_df = test_df[
    test_df[TEXT_COL].ne("")
].copy()


test_df.reset_index(
    drop=True,
    inplace=True
)


print("\nTest samples:", len(test_df))

print("\nTest class distribution:")

print(

    test_df[RATE_COL]

    .value_counts()

    .reindex(["low", "medium", "high"], fill_value=0)

)


# ============================================================
# 21. Check Exact Text Overlap
# ============================================================

print("\n" + "=" * 65)
print("CHECKING TRAIN / TEST TEXT OVERLAP")
print("=" * 65)


train_texts = set(

    data[TEXT_COL]

    .dropna()

    .astype(str)

    .str.strip()

)


test_texts = set(

    test_df[TEXT_COL]

    .dropna()

    .astype(str)

    .str.strip()

)


overlap_texts = train_texts.intersection(
    test_texts
)


print("Unique training texts:", len(train_texts))

print("Unique test texts:", len(test_texts))

print("Overlapping unique texts:", len(overlap_texts))


if len(overlap_texts) > 0:

    overlap_ratio = (

        len(overlap_texts) /

        max(len(test_texts), 1)

    )

    print(

        "WARNING: Exact text overlap detected."

    )

    print(

        "Test overlap ratio:",

        round(overlap_ratio, 6)

    )

    print(

        "Consider investigating this before reporting final results."

    )

else:

    print("No exact text overlap detected.")


# ============================================================
# 22. Final Independent Test Evaluation
# ============================================================

print("\n" + "=" * 65)
print("FINAL TEST EVALUATION")
print("=" * 65)


final_test_dataset = RatingDataset(

    test_df,

    tokenizer,

    include_labels=True

)


test_results = trainer_stage2.predict(
    final_test_dataset
)


test_logits = test_results.predictions

y_true = test_df[LABEL_COL].to_numpy()

y_pred = np.argmax(
    test_logits,
    axis=1
)


# ------------------------------------------------------------
# Classification metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_true,
    y_pred
)


precision_macro, recall_macro, f1_macro, _ = (
    precision_recall_fscore_support(

        y_true,

        y_pred,

        average="macro",

        zero_division=0

    )
)


precision_weighted, recall_weighted, f1_weighted, _ = (
    precision_recall_fscore_support(

        y_true,

        y_pred,

        average="weighted",

        zero_division=0

    )
)


cm = confusion_matrix(

    y_true,

    y_pred,

    labels=[0, 1, 2]

)


report = classification_report(

    y_true,

    y_pred,

    labels=[0, 1, 2],

    target_names=[

        "low",

        "medium",

        "high"

    ],

    output_dict=True,

    zero_division=0

)


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("\nFINAL TEST RESULTS")

print("-" * 50)

print("Accuracy:", round(accuracy, 6))

print("Macro Precision:", round(precision_macro, 6))

print("Macro Recall:", round(recall_macro, 6))

print("Macro F1:", round(f1_macro, 6))

print("Weighted Precision:", round(precision_weighted, 6))

print("Weighted Recall:", round(recall_weighted, 6))

print("Weighted F1:", round(f1_weighted, 6))


print("\nClassification Report:")

print(

    classification_report(

        y_true,

        y_pred,

        labels=[0, 1, 2],

        target_names=[

            "low",

            "medium",

            "high"

        ],

        zero_division=0

    )

)


print("\nConfusion Matrix:")

print(cm)


# ============================================================
# 23. Save Predictions
# ============================================================

test_probabilities = torch.softmax(

    torch.tensor(test_logits),

    dim=-1

).numpy()


predictions_df = test_df.copy()


predictions_df["true_label"] = (

    predictions_df[LABEL_COL]

    .map(ID2LABEL)

)


predictions_df["predicted_label_id"] = y_pred


predictions_df["predicted_label"] = (

    predictions_df["predicted_label_id"]

    .map(ID2LABEL)

)


predictions_df["confidence"] = (

    np.max(test_probabilities, axis=1)

)


for class_id, class_name in ID2LABEL.items():

    predictions_df[

        f"probability_{class_name}"

    ] = test_probabilities[:, class_id]


predictions_df["correct"] = (

    predictions_df["true_label"]

    == predictions_df["predicted_label"]

)


predictions_df.to_csv(

    os.path.join(

        EVALUATION_DIR,

        "final_test_predictions.csv"

    ),

    index=False,

    encoding="utf-8-sig"

)


# ============================================================
# 24. Save Confusion Matrix
# ============================================================

confusion_df = pd.DataFrame(

    cm,

    index=["true_low", "true_medium", "true_high"],

    columns=["pred_low", "pred_medium", "pred_high"]

)


confusion_df.to_csv(

    os.path.join(

        EVALUATION_DIR,

        "confusion_matrix.csv"

    ),

    encoding="utf-8-sig"

)


# ============================================================
# 25. Save Final Metrics
# ============================================================

final_metrics = {

    "model": MODEL_NAME,

    "transformers_version": __import__("transformers").__version__,

    "max_length": MAX_LENGTH,

    "confidence_threshold": CONFIDENCE_THRESHOLD,

    "mask_rate": MASK_RATE,

    "train_samples": int(len(supervised_train)),

    "masked_samples": int(len(masked_train)),

    "validation_samples": int(len(validation_df)),

    "natural_unlabeled_samples": int(len(natural_unlabeled)),

    "pseudo_candidates": int(len(pseudo_candidates)),

    "accepted_pseudo_labels": int(len(accepted_pseudo)),

    "stage2_training_samples": int(len(stage2_train_df)),

    "test_samples": int(len(test_df)),

    "test_accuracy": float(accuracy),

    "test_macro_precision": float(precision_macro),

    "test_macro_recall": float(recall_macro),

    "test_macro_f1": float(f1_macro),

    "test_weighted_precision": float(precision_weighted),

    "test_weighted_recall": float(recall_weighted),

    "test_weighted_f1": float(f1_weighted),

    "confusion_matrix": cm.tolist(),

    "classification_report": report,

    "stage1_validation_metrics": stage1_validation_metrics,

    "stage2_validation_metrics": stage2_validation_metrics,

    "exact_text_overlap_count": int(len(overlap_texts))

}


with open(

    os.path.join(

        EVALUATION_DIR,

        "final_metrics.json"

    ),

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        final_metrics,

        f,

        ensure_ascii=False,

        indent=4

    )


# ============================================================
# 26. Save Final Model
# ============================================================

FINAL_MODEL_DIR = os.path.join(

    OUTPUT_ROOT,

    "final_model"

)

trainer_stage2.save_model(
    FINAL_MODEL_DIR
)

tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)


# ============================================================
# 27. Final Summary
# ============================================================

print("\n" + "=" * 65)

print("PIPELINE COMPLETED SUCCESSFULLY")

print("=" * 65)


print("\nFinal model:", FINAL_MODEL_DIR)

print("Evaluation directory:", EVALUATION_DIR)

print("Pseudo-label directory:", PSEUDO_DIR)

print("Split directory:", SPLIT_DIR)


print("\nFinal Macro F1:", round(f1_macro, 6))

print("Final Accuracy:", round(accuracy, 6))

print("Final Weighted F1:", round(f1_weighted, 6))


print("\nSaved files:")

for root, dirs, files in os.walk(OUTPUT_ROOT):

    for filename in files:

        print(
            os.path.join(root, filename)
        )


print("\nALL DONE.")


W1111 08:43:12.801000 21692 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


DIGIKALA PARSBERT SEMI-SUPERVISED TRAINING
Device: cuda
Transformers version: 5.21.0
GPU: NVIDIA GeForce RTX 3060 Laptop GPU

LOADING TRAINING DATA
Original shape: (995816, 15)

Labeled samples: 600000
Naturally unlabeled samples: 395816

Labeled class distribution:
labels
low       200000
medium    200000
high      200000
Name: count, dtype: int64

TRAIN / VALIDATION SPLIT
Train pool: 540000
Validation: 60000

Train pool distribution:
labels
low       180000
medium    180000
high      180000
Name: count, dtype: int64

Validation distribution:
labels
low       20000
medium    20000
high      20000
Name: count, dtype: int64

DETERMINISTIC LABEL MASKING

Mask summary:
{
    "low": {
        "original_count": 180000,
        "masked_count": 36000,
        "remaining_labeled": 144000,
        "actual_mask_rate": 0.2
    },
    "medium": {
        "original_count": 180000,
        "masked_count": 36000,
        "remaining_labeled": 144000,
        "actual_mask_rate": 0.2
    },
    "high": 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: HooshvareLab/bert-base-parsbert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING

Step,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
10000,0.751885,0.742626,0.718217,0.715904,0.715904
20000,0.712361,0.742491,0.719767,0.719014,0.719014
30000,0.681777,0.751894,0.719800,0.718801,0.718801
40000,0.672105,0.748940,0.722200,0.717614,0.717614
40500,0.674326,0.748873,0.722467,0.718282,0.718282


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (15 outputs left out)


Stage 2 Validation Metrics:
{'eval_loss': 0.7662107944488525, 'eval_accuracy': 0.7125666666666667, 'eval_macro_f1': 0.7083249827546855, 'eval_weighted_f1': 0.7083249827546855}

LOADING INDEPENDENT TEST DATASET
Original test shape: (89814, 7)
Test columns: ['id', 'text', 'recommendation_status', 'my_label', 'mismatch', 'rate_score', 'rate']

Test samples: 89814

Test class distribution:
rate
low       25479
medium    37059
high      27276
Name: count, dtype: Int64

CHECKING TRAIN / TEST TEXT OVERLAP
Unique training texts: 995816
Unique test texts: 89814
Overlapping unique texts: 0
No exact text overlap detected.

FINAL TEST EVALUATION



FINAL TEST RESULTS
--------------------------------------------------
Accuracy: 0.6936
Macro Precision: 0.695277
Macro Recall: 0.718015
Macro F1: 0.698405
Weighted Precision: 0.695245
Weighted Recall: 0.6936
Weighted F1: 0.685816

Classification Report:
              precision    recall  f1-score   support

         low       0.65      0.83      0.73     25479
      medium       0.69      0.52      0.59     37059
        high       0.75      0.80      0.78     27276

    accuracy                           0.69     89814
   macro avg       0.70      0.72      0.70     89814
weighted avg       0.70      0.69      0.69     89814


Confusion Matrix:
[[21250  4028   201]
 [10875 19092  7092]
 [  630  4693 21953]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


PIPELINE COMPLETED SUCCESSFULLY

Final model: digikala_parsbert_semi_supervised\final_model
Evaluation directory: digikala_parsbert_semi_supervised\evaluation
Pseudo-label directory: digikala_parsbert_semi_supervised\pseudo_labels
Split directory: digikala_parsbert_semi_supervised\splits

Final Macro F1: 0.698405
Final Accuracy: 0.6936
Final Weighted F1: 0.685816

Saved files:
digikala_parsbert_semi_supervised\evaluation\confusion_matrix.csv
digikala_parsbert_semi_supervised\evaluation\final_metrics.json
digikala_parsbert_semi_supervised\evaluation\final_test_predictions.csv
digikala_parsbert_semi_supervised\final_model\config.json
digikala_parsbert_semi_supervised\final_model\model.safetensors
digikala_parsbert_semi_supervised\final_model\tokenizer.json
digikala_parsbert_semi_supervised\final_model\tokenizer_config.json
digikala_parsbert_semi_supervised\final_model\training_args.bin
digikala_parsbert_semi_supervised\pseudo_labels\accepted_pseudo_labels.csv
digikala_parsbert_semi_supe